# my notebook FINAL v3 (USE THIS ONE!!)

> ⚠️ **This notebook is an intentional ANTI-EXAMPLE for the MLOps course.**
> Every cell below demonstrates one or more things you should **never** do.
> The full list of problems — and how to convert this into production-ready
> scripts — is in the [README](README.md#-what-not-to-do-the-everything-notebook).

ride duration prediction. eda + preprocessing + training + evaluation + tests, all here :)

In [ ]:
# ❌ no versions pinned — whoever runs this next month gets DIFFERENT library versions
# ❌ 'sklearn' is even the wrong (deprecated) package name — should be scikit-learn
# ❌ installing from inside the notebook mutates the environment as a side effect
!pip install pandas numpy sklearn matplotlib seaborn xgboost

In [ ]:
import pandas as pd
import numpy as np
import pickle
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import *          # ❌ wildcard import — nobody knows what's in scope
from sklearn.ensemble import *              # ❌ same, and it can shadow names silently
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')           # ❌ hide every warning, incl. the ones that matter

# ❌❌❌ SECRETS HARDCODED IN A NOTEBOOK — this gets committed to git and leaked
MLFLOW_TRACKING_TOKEN = "sk-live-8f2a9b1c-SUPER-SECRET-do-not-share"
DB_PASSWORD = "admin123"

%matplotlib inline

In [ ]:
# ============ LOAD DATA ============
# ❌ hardcoded ABSOLUTE path to one person's laptop — breaks on every other machine and in CI
try:
    df = pd.read_csv("/Users/aya/Desktop/data/taxi_rides_FINAL_v3_use_this.csv")
except:
    # ❌ bare except that silently swallows the error and fabricates random data instead.
    # If the file is missing you'd want a loud failure — instead you train on garbage
    # and never notice. Also: ❌ no random seed, so every run gives a different dataset.
    df = pd.DataFrame({
        "distance": np.random.uniform(0.5, 30, 1000),          # miles?? km?? who knows
        "passengers": np.random.randint(1, 5, 1000),
        "pickup_hour": np.random.randint(0, 24, 1000),
        "duration": None,
    })
    df["duration"] = df["distance"] * 2.1 + df["passengers"] * 1.5 + np.random.normal(0, 3, 1000)
    print("couldnt find the file, using random data instead lol")

print(df.shape)
df.head()

In [ ]:
# ============ EDA ============
# ❌ exploration mixed into the same notebook as the training pipeline —
# every "production" run re-renders plots nobody looks at
print(df.describe())
print("----------")
print(df.isna().sum())
print("----------")
print("correlation!!!")
print(df.corr())

plt.figure(figsize=(6, 4))
sns.histplot(df["duration"])
plt.title("duration histogram")
plt.show()

sns.scatterplot(x=df["distance"], y=df["duration"])
plt.show()

# ❌ shadowing Python builtins — `list` and `sum` are now broken for the rest of the notebook
list = df.columns.tolist()
sum = df["duration"].sum()
print(list, sum)

In [ ]:
# ============ PREPROCESSING ============
# ❌ NON-IDEMPOTENT cell: it mutates df IN PLACE. Run it twice and the distance
# gets converted miles→km TWICE (×1.60934 ×1.60934). Classic hidden-notebook-state bug.
df["distance"] = df["distance"] * 1.60934   # convert to km (i think the data is miles? not sure)

# ❌ silently dropping rows — no record of how many, no data-validation rules
df = df.dropna()

# ❌ magic numbers with no explanation, hardcoded inline
df = df[df["duration"] < 120]
df = df[df["distance"] > 0.3]
df = df[df["passengers"] <= 4]

# ❌ DATA LEAKAGE #1: feature built FROM THE TARGET — the model just reads the answer
df["duration_per_km"] = df["duration"] / df["distance"]

print("after cleaning:", df.shape)   # ❌ print-debugging instead of logging

In [ ]:
# ❌ DATA LEAKAGE #2: scaler is fit on the WHOLE dataset BEFORE the train/test split,
# so test-set statistics leak into training
scaler = StandardScaler()
features = ["distance", "passengers", "pickup_hour", "duration_per_km"]
df[features] = scaler.fit_transform(df[features])

X = df[features]
y = df["duration"]

# ❌ no random_state → a different split every run, results not reproducible
# ❌ 0.33 is another unexplained magic number
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)
print(len(X_train), len(X_test))

In [ ]:
# ============ TRAINING ============
# ❌ copy-pasted model variants instead of a config/loop — df2, model2, model_new...
# ❌ model selection is done ON THE TEST SET (peeking) — the test set is now burned
model = LinearRegression()
model.fit(X_train, y_train)
print("lr test r2:", model.score(X_test, y_test))

model2 = Ridge(alpha=0.1)   # ❌ tried alpha=1, 0.5, 0.3 by hand-editing this cell, history lost
model2.fit(X_train, y_train)
print("ridge test r2:", model2.score(X_test, y_test))

# model3 = Lasso(alpha=0.01)                     # ❌ dead code kept as "version control"
# model3.fit(X_train, y_train)
# print("lasso:", model3.score(X_test, y_test))  # this one was bad, 0.62 i think??

model_new = RandomForestRegressor(n_estimators=100)   # ❌ no seed here either
model_new.fit(X_train, y_train)
print("rf test r2:", model_new.score(X_test, y_test))

# ❌ "best" chosen manually by reading the prints above and editing this line:
best_model = model_new   # rf was best when I ran it on tuesday

In [ ]:
# ============ EVALUATION ============
# ❌ reports metrics ON THE TRAINING SET and calls it "model accuracy"
preds = best_model.predict(X_train)
print("MODEL ACCURACY (r2):", r2_score(y_train, preds))          # looks amazing, means nothing
print("rmse:", np.sqrt(mean_squared_error(y_train, preds)))

# ❌ metrics only exist as print output — no experiment tracking, no history,
# no way to compare this run to last week's run
# rf tuesday: 0.97   <- ❌ results "tracked" in comments
# rf monday: 0.95 (or 0.94?)
# ridge: 0.81

In [ ]:
# ============ PREDICT ON NEW DATA ============
# ❌ TRAINING/SERVING SKEW: preprocessing copy-pasted from above but subtly DIFFERENT —
# forgot the miles→km conversion AND builds duration_per_km from a guess (the real
# target obviously doesn't exist at inference time... because leakage feature)
new_rides = pd.DataFrame({
    "distance": [5.2, 12.0],
    "passengers": [1, 3],
    "pickup_hour": [9, 18],
})
new_rides["duration_per_km"] = 2.1   # "should be close enough"
# ❌ reuses the notebook-global `scaler` — works here, impossible to reproduce in an API
new_scaled = scaler.transform(new_rides[["distance", "passengers", "pickup_hour", "duration_per_km"]])
print("predictions:", best_model.predict(new_scaled))

In [ ]:
# ============ SAVE MODEL ============
# ❌ pickle to a hardcoded path on someone's Desktop
# ❌ "versioning" via filename: final_v2_REAL_final — no registry, no metadata,
#    no record of data/params/metrics that produced it
# ❌ the scaler is NOT saved — anyone loading this model can't reproduce the features
pickle.dump(best_model, open("/Users/aya/Desktop/model_final_v2_REAL_final.pkl", "wb"))
print("saved!!")

In [ ]:
# ============ TESTS ============
# ❌ "tests" living as notebook cells: run manually, never in CI, gone on restart

# ❌ test 1: always passes, tests literally nothing
assert True
print("test 1 passed ✅")

# ❌ test 2: asserts on TRAINING accuracy with an arbitrary threshold —
# an overfit model passes this proudly
assert best_model.score(X_train, y_train) > 0.5, "model is bad"
print("test 2 passed ✅")

# ❌ test 3: hardcoded expected value from ONE unseeded run — flaky by construction,
# fails or passes depending on the day
pred = best_model.predict(X_train.iloc[[0]])[0]
# assert abs(pred - 23.7) < 0.01   # kept failing so I commented it out ¯\\_(ツ)_/¯

# ❌ test 4: swallows the failure and reports success anyway
try:
    assert len(X_test) > 100000   # obviously false
except:
    pass
print("all tests passed ✅✅✅")

In [ ]:
# ❌ OUT-OF-ORDER DEPENDENCY: this cell only works if you ALREADY ran the cell below it.
# (Classic notebook: "run cell 14 first, then come back here" — undocumented, of course.)
print("final report:", REPORT_TITLE)
print("model:", type(best_model).__name__)

In [ ]:
# ...defined here, AFTER the cell that uses it. Restart-and-run-all → NameError.
REPORT_TITLE = "ride duration model - final report v3"

## notes

- TODO: clean this up later
- TODO: ask ahmed which csv is the right one
- if it doesnt work restart the kernel and run the cells in a different order until it does
- **See the [README](README.md#-what-not-to-do-the-everything-notebook) for every problem in this notebook and how the rest of `session_1/` fixes each one.**